In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.5 MB/s eta 0:00:00


In [5]:
import numpy as np
import cv2
import time
import gdown
from ultralytics import YOLO

from google.colab.patches import cv2_imshow

import time

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
file_id = "1IhGmwJA48qlg5tKtrieTP7UGgwF3WUYC"
url = f"https://drive.google.com/uc?id={file_id}"


gdown.download(url, output="test_video.mp4", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1IhGmwJA48qlg5tKtrieTP7UGgwF3WUYC
To: /content/test_video.mp4
100%|██████████| 53.4M/53.4M [00:01<00:00, 39.0MB/s]


'test_video.mp4'

In [7]:
file_id = "1IhGmwJA48qlg5tKtrieTP7UGgwF3WUYC"
url = f"https://drive.google.com/uc?id={file_id}"


gdown.download(url, output="test_video.mp4", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1IhGmwJA48qlg5tKtrieTP7UGgwF3WUYC
To: /content/test_video.mp4
100%|██████████| 53.4M/53.4M [00:00<00:00, 96.7MB/s]


'test_video.mp4'

In [8]:
file_id = "14DBG8KkxsaxhIs95VI_eF4TkrM3LnajP"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, output="YOLO_50.pt", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=14DBG8KkxsaxhIs95VI_eF4TkrM3LnajP
To: /content/YOLO_50.pt
100%|██████████| 19.2M/19.2M [00:00<00:00, 49.3MB/s]


'YOLO_50.pt'

In [9]:
model_path = "/content/YOLO_50.pt"
video_path = "/content/test_video.mp4"

In [10]:
def ball_detector(model_path, video_path):

  skiped_frame = 0
  valid_frame = 0
  all_frame = 0

  model = YOLO(model_path)

  ball_cords = {}

  start = time.time()

  results = model.predict(source = video_path,
                         stream = True)

  for frame_id, result in enumerate(results):
    all_frame += 1

    if len(result.boxes) == 0:
      skiped_frame += 1
      continue

    boxes = result.boxes.xywh.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()

    best_box = np.argmax(scores)

    x,y,w,h = boxes[best_box]

    end = time.time()

    ball_cords[frame_id] = {"x": float(x),
                            "y": float(y),
                            "w": float(w),
                            "h": float(h),
                            "scores": float(scores[best_box])}
    valid_frame += 1

  time_ = end - start
  timing = (all_frame / 30)
  print(f"Всего фреймов:{all_frame}")
  print(f"Успешно обработанных фреймов:{valid_frame}")
  print(f"Пропущено фреймов:{skiped_frame}")
  print(f"На обраотку {timing} секунду видео затрачено {time_} секунд")

  return ball_cords

In [11]:
ball_cords = ball_detector(model_path, video_path)

Выходные данные были обрезаны до нескольких последних строк (5000).
video 1/1 (frame 1396/6420) /content/test_video.mp4: 736x1280 (no detections), 16.2ms
video 1/1 (frame 1397/6420) /content/test_video.mp4: 736x1280 (no detections), 16.0ms
video 1/1 (frame 1398/6420) /content/test_video.mp4: 736x1280 (no detections), 17.7ms
video 1/1 (frame 1399/6420) /content/test_video.mp4: 736x1280 (no detections), 16.8ms
video 1/1 (frame 1400/6420) /content/test_video.mp4: 736x1280 (no detections), 16.3ms
video 1/1 (frame 1401/6420) /content/test_video.mp4: 736x1280 (no detections), 18.0ms
video 1/1 (frame 1402/6420) /content/test_video.mp4: 736x1280 (no detections), 17.6ms
video 1/1 (frame 1403/6420) /content/test_video.mp4: 736x1280 (no detections), 16.4ms
video 1/1 (frame 1404/6420) /content/test_video.mp4: 736x1280 (no detections), 17.5ms
video 1/1 (frame 1405/6420) /content/test_video.mp4: 736x1280 (no detections), 17.5ms
video 1/1 (frame 1406/6420) /content/test_video.mp4: 736x1280 (no detect

In [12]:
import json
with open("ball_cords.json", "w") as f:
    json.dump(ball_cords, f)

In [13]:
with open("ball_cords.json", "r") as f:
    data = json.load(f)

sorted_data = {int(k): data[k] for k in sorted(data, key=lambda x: int(x))}

with open("ball_cords_sorted.json", "w") as f:
    json.dump(sorted_data, f, indent=4)

In [14]:
print(f"Фактическая длина ball_cords: {len(ball_cords)}")

Фактическая длина ball_cords: 1049
